In [1]:
!pip install rdkit
# !pip install e3fp
# !pip install map4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 41.5 MB/s eta 0:00:00


In [2]:
# from e3fp.pipeline import fprints_from_smiles

In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.AtomPairs import Pairs, Torsions
from rdkit.Avalon import pyAvalonTools
from sklearn.preprocessing import StandardScaler
import pickle
from tqdm.notebook import tqdm

In [4]:
# File paths
file_paths = {
    'train': '/kaggle/input/scaffold-split/scaffold_train_dataset.csv',
    'validation': '/kaggle/input/scaffold-split/scaffold_validation_dataset.csv',
    'test': '/kaggle/input/scaffold-split/scaffold_test_dataset.csv'
}


In [5]:
# Function to read data and create RDKit molecules
def read_data(filepath):
    print(f"Reading {filepath}...")
    df = pd.read_csv(filepath)
    
    # Create RDKit molecules from SMILES
    molecules = []
    valid_indices = []
    smiles_str = []
    for idx, smiles in enumerate(tqdm(df['SMILES'], desc="Parsing SMILES")):
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            molecules.append(mol)
            valid_indices.append(idx)
            smiles_str.append(smiles)
        else:
            print(f"Invalid SMILES: {smiles}")
    
    # Keep only valid molecules
    df_valid = df.iloc[valid_indices].reset_index(drop=True)
    
    return df_valid, molecules, smiles_str

In [6]:
# Function to generate Morgan fingerprints
def generate_morgan_fps(molecules, radius=2, nBits=2048):
    print("Generating Morgan fingerprints...")
    # Preallocate array for all fingerprints
    morgan_fps = np.empty((len(molecules), nBits), dtype=int)
    # Initialize the generator once
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
    
    for i, mol in enumerate(tqdm(molecules, desc="Morgan FPs")):
        fp = mfpgen.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, morgan_fps[i])
    
    return morgan_fps

# Function to generate RDKit topological fingerprints
def generate_topological_fps(molecules, nBits=2048):
    print("Generating Topological fingerprints...")
    topo_fps = np.empty((len(molecules), nBits), dtype=int)
    # Initialize the topological torsion generator once
    ttgen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=nBits)
    
    for i, mol in enumerate(tqdm(molecules, desc="Topological FPs")):
        fp = ttgen.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, topo_fps[i])
    
    return topo_fps

# Function to generate MACCS keys
def generate_maccs_fps(molecules):
    print("Generating MACCS fingerprints...")
    # MACCS fingerprints typically have a length of 167 bits
    maccs_length = 167
    maccs_fps = np.empty((len(molecules), maccs_length), dtype=int)
    
    for i, mol in enumerate(tqdm(molecules, desc="MACCS FPs")):
        fp = AllChem.GetMACCSKeysFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, maccs_fps[i])
    
    return maccs_fps

# Function to generate Atom Pair fingerprints
def generate_atom_pair_fps(molecules, nBits=2048):
    print("Generating Atom Pair fingerprints...")
    atom_pair_fps = np.empty((len(molecules), nBits), dtype=int)
    # Initialize the atom pair generator once
    apgen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=nBits)
    
    for i, mol in enumerate(tqdm(molecules, desc="Atom Pair FPs")):
        fp = apgen.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, atom_pair_fps[i])
    
    return atom_pair_fps

# Function to generate Avalon fingerprints
def generate_avalon_fps(molecules, nBits=2048):
    print("Generating Avalon fingerprints...")
    avalon_fps = np.empty((len(molecules), nBits), dtype=int)
    
    for i, mol in enumerate(tqdm(molecules, desc="Avalon FPs")):
        fp = pyAvalonTools.GetAvalonFP(mol, nBits=nBits)
        DataStructs.ConvertToNumpyArray(fp, avalon_fps[i])
    
    return avalon_fps

# def generate_e3fp_fps(smiles, name="e3fp"):
#     print("Generating E3FP fingerprints...")
    
#     fps = []
#     for i, sm in enumerate(tqdm(smiles, desc="E3FP FPs")):
#         try:
#             # Process one SMILES at a time
#             fp = fprints_from_smiles(sm, name = f'{name}_i')
#             # Convert the fingerprint to a NumPy array (if necessary)
#             fps.append(np.array(fp))
#         except Exception as e:
#             print(f"Error generating E3FP fingerprint for {sm}: {e}")
#             # Optionally, handle the error by appending a zero vector or skipping
#             fps.append(np.zeros(1))
#     return np.array(fps)


In [7]:
# Process each dataset
datasets = {}
molecules = {}
smiles = {}
for dataset_type, file_path in file_paths.items():
    df, mols, smile = read_data(file_path)
    datasets[dataset_type] = df
    molecules[dataset_type] = mols
    smiles[dataset_type] = smile
    print(f"Processed {dataset_type} dataset: {len(df)} valid molecules")

# Generate all fingerprints for each dataset
fingerprints = {}

for dataset_type in datasets.keys():
    print(f"\nProcessing fingerprints for {dataset_type} dataset")
    df = datasets[dataset_type]
    mols = molecules[dataset_type]
    smile = smiles[dataset_type]
    # print(smile)
    fingerprints[dataset_type] = {
        'morgan': generate_morgan_fps(mols),
         'topological': generate_topological_fps(mols),
         'maccs': generate_maccs_fps(mols),
         'atom_pair': generate_atom_pair_fps(mols),
         'avalon': generate_avalon_fps(mols),
        # , 'e3fp': generate_e3fp_fps(smile, name="e3fp")
    }

Reading /kaggle/input/scaffold-split/scaffold_train_dataset.csv...


Parsing SMILES:   0%|          | 0/1650 [00:00<?, ?it/s]

Processed train dataset: 1650 valid molecules
Reading /kaggle/input/scaffold-split/scaffold_validation_dataset.csv...


Parsing SMILES:   0%|          | 0/300 [00:00<?, ?it/s]

Processed validation dataset: 300 valid molecules
Reading /kaggle/input/scaffold-split/scaffold_test_dataset.csv...


Parsing SMILES:   0%|          | 0/335 [00:00<?, ?it/s]

... (26 outputs left out)

Generating Atom Pair fingerprints...


Atom Pair FPs:   0%|          | 0/335 [00:00<?, ?it/s]

Generating Avalon fingerprints...


Avalon FPs:   0%|          | 0/335 [00:00<?, ?it/s]

In [8]:
final_datasets = {}

for dataset_type in datasets.keys():
    df = datasets[dataset_type]  # Original molecule info DataFrame
    fps = fingerprints[dataset_type]  # Dict of fingerprint arrays
    
    # Create a DataFrame with the molecule info
    final_df = df[['Molecule', 'SMILES', 'Affinity', 'weight']].copy()
    
    # Add each fingerprint type to the DataFrame
    for fp_type, fp_data in fps.items():
        fp_df = pd.DataFrame(
            fp_data, 
            columns=[f"{fp_type}_{i}" for i in range(fp_data.shape[1])]
        )
        final_df = pd.concat([final_df, fp_df], axis=1)
    
    final_df.to_csv(f'scaffold_{dataset_type}_fp.csv', index=False)
    final_datasets[dataset_type] = final_df
    print(f"{dataset_type} dataset shape: {final_df.shape}")


train dataset shape: (1650, 8363)
validation dataset shape: (300, 8363)
test dataset shape: (335, 8363)
